Siamese Network

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import random
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Lambda, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
import os

# Load handcrafted features
data = pd.read_csv("../Dataset/Features/signature_features.csv")

# Extract file paths and labels
labels = data["label"].values
image_paths = [f"../Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{file}"
               for file, label in zip(data["filename"], labels)]

# Split dataset into train and validation
train_img_paths, val_img_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels)

# Image size for InceptionV3
image_size = (299, 299)


# 🔹 Custom Data Generator for Siamese Network
class SiameseDataGenerator(Sequence):
    def __init__(self, image_paths, labels, batch_size=32, shuffle=True):
        self.image_paths = image_paths
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.image_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_pairs = []
        batch_labels = []

        for _ in range(self.batch_size):
            is_genuine = random.choice([True, False])  # Randomly choose whether to pair genuine or forged
            idx1 = random.randint(0, len(self.image_paths) - 1)
            img1_path = self.image_paths[idx1]

            if is_genuine:
                # Pair with another genuine signature
                same_class_indices = [i for i in range(len(self.image_paths)) if self.labels[i] == self.labels[idx1]]
                idx2 = random.choice(same_class_indices)
            else:
                # Pair with a forged signature
                different_class_indices = [i for i in range(len(self.image_paths)) if self.labels[i] != self.labels[idx1]]
                idx2 = random.choice(different_class_indices)

            img2_path = self.image_paths[idx2]
            label = 1 if not is_genuine else 0  # 1 = Forged, 0 = Genuine

            batch_pairs.append((img1_path, img2_path))
            batch_labels.append(label)

        # Load images
        batch_images1 = np.array([self.load_image(p1) for p1, _ in batch_pairs])
        batch_images2 = np.array([self.load_image(p2) for _, p2 in batch_pairs])
        batch_labels = np.array(batch_labels)

        return (np.array(batch_images1), np.array(batch_images2)), np.array(batch_labels)


    def on_epoch_end(self):
        if self.shuffle:
            indices = np.arange(len(self.image_paths))
            np.random.shuffle(indices)
            self.image_paths = [self.image_paths[i] for i in indices]
            self.labels = [self.labels[i] for i in indices]

    @staticmethod
    def load_image(image_path):
        img = tf.keras.preprocessing.image.load_img(image_path, target_size=image_size)
        img = tf.keras.preprocessing.image.img_to_array(img)
        return tf.keras.applications.inception_v3.preprocess_input(img)


# Create data generators
batch_size = 32
train_generator = SiameseDataGenerator(train_img_paths, train_labels, batch_size=batch_size)
val_generator = SiameseDataGenerator(val_img_paths, val_labels, batch_size=batch_size)


# 🔹 Siamese Network: Shared CNN Feature Extractor
def build_base_network(input_shape):
    base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=input_shape)
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.5)(x)
    model = Model(base_model.input, x, name="Shared_CNN")
    return model


# Create shared CNN model
input_shape = (299, 299, 3)
base_network = build_base_network(input_shape)

# Siamese Network Inputs
input_a = Input(shape=input_shape)
input_b = Input(shape=input_shape)

# Feature Extraction (Shared Weights)
feat_a = base_network(input_a)
feat_b = base_network(input_b)

# Compute Euclidean Distance between embeddings
def euclidean_distance(vects):
    x, y = vects
    return K.sqrt(K.sum(K.square(x - y), axis=1, keepdims=True))

distance = Lambda(euclidean_distance)([feat_a, feat_b])

# Output Layer (Binary Classification)
output = Dense(1, activation="sigmoid")(distance)

# Define Siamese Model
siamese_model = Model(inputs=[input_a, input_b], outputs=output)

# Compile Model
siamese_model.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=0.0001), metrics=["accuracy"])

# 🔹 Train the Model
siamese_model.fit(train_generator, validation_data=val_generator, epochs=25)

# Save the Model
siamese_model.save("../Model/signature_siamese_model.h5")

print("Siamese Model training completed and saved!")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Load the trained Siamese Model
siamese_model = tf.keras.models.load_model("../Model/signature_siamese_model.h5", compile=False)

# Load validation data
val_generator = SiameseDataGenerator(val_img_paths, val_labels, batch_size=32, shuffle=False)

# Get true labels and predictions
y_true = []
y_pred = []

for (img1_batch, img2_batch), labels in val_generator:
    predictions = siamese_model.predict([img1_batch, img2_batch]).flatten()
    y_pred.extend(predictions)
    y_true.extend(labels)

# Convert predictions to binary values (Threshold = 0.5)
y_pred_binary = [1 if p > 0.5 else 0 for p in y_pred]

# Compute Metrics
accuracy = accuracy_score(y_true, y_pred_binary)
precision = precision_score(y_true, y_pred_binary)
recall = recall_score(y_true, y_pred_binary)
f1 = f1_score(y_true, y_pred_binary)
conf_matrix = confusion_matrix(y_true, y_pred_binary)

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)
